# Process v4 stop-level delay data for the dashboard

Reads the v2 aggregator outputs (`stop_metrics_v2.gpkg` — the `routes`
layer and `stop_metrics_weekday_v2.csv` / `stop_metrics_weekend_v2.csv`)
and writes the same two JSON files used by `js/delay.js`:

- **`network_routes.json`**
- **`route_details.json`**

## What changed since v3

* **Outlier-robust route median** — v3 reported the route's
  arrivals-weighted *arithmetic mean* of per-stop medians, which a
  single stop with a 1440-min missed-trip entry could blow up. v4
  reports the **arrivals-weighted median** of per-stop medians, and
  the JSON key is renamed `mean` → `median` so the dashboard label
  reads honestly.

## What changed since v3

* **Outlier-robust route median** — v3 reported the route's
  arrivals-weighted *arithmetic mean* of per-stop medians, which a
  single stop with a 1440-min missed-trip entry could blow up. v4
  reports the **arrivals-weighted median** of per-stop medians, and
  the JSON key is renamed `mean` → `median` so the dashboard label
  reads honestly.

## What changed since v3

* **Both directions exported** — v3 collapsed each `route_short_name` to
  its busiest direction, but the routed geometry is direction-specific
  (one-way streets, dual carriageways) so picking one was misleading.
  v4 emits one record per `(route_short_name, direction_id)`. Each
  record carries:
    * `key`         — `"<short_name>-<direction_id>"`, unique per record
    * `name`        — `route_short_name` (shared by the two directions)
    * `direction_id`, `route_id`
    * `direction`   — `"<first stop> → <last stop>"`, human-readable
    * `terminus_from`, `terminus_to`
* **No-service periods now read `null`, not `0.0`** — when a route has
  no observed arrivals in a period (e.g. AM off-peak on a peak-only
  service), `weighted_period_stats` used to return
  `{mean: 0.0, otp: 0.0, n: 0}` so the league row displayed "0% on-time"
  for what was actually "no service". v4 returns
  `{mean: None, otp: None, n: 0}`; `delay.js` renders these as `—`.

## What changed since v2

* **`path` per route** — `[[lat, lon], …]` along the canonical
  direction's LineString, simplified to `PATH_SIMPLIFY_TOL` degrees
  and rounded to `PATH_COORD_DECIMALS` decimal places.
* **MIN_ARRIVALS_PER_ROUTE filter** — drops `(route, direction)` pairs
  with too few weekday arrivals to be statistically meaningful.

To keep the payload manageable, at most `MAX_TRIPS_PER_PERIOD` raw
delay observations are kept per stop / period (default 12).

In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString

# ── Inputs ─────────────────────────────────────────────────────────
PROJ_DIR  = Path(r"D:\2026_03-Bus_Project")
INPUT_DIR = PROJ_DIR / "output" / "aggregate"
STOP_METRICS_GPKG = INPUT_DIR / "stop_metrics_v2.gpkg"
WEEKDAY_CSV       = INPUT_DIR / "stop_metrics_weekday_v2.csv"
WEEKEND_CSV       = INPUT_DIR / "stop_metrics_weekend_v2.csv"

# ── Output ─────────────────────────────────────────────────────────
OUTPUT_DIR = Path('.').resolve()

# ── Filters (unchanged from v2) ──────────────────────────────────────
N_ROUTES = None
ALLOWED_ROUTE_TYPES = ["Bus"]
MIN_STOPS_PER_ROUTE = 4
# Drop routes whose total weekday `n_arrivals_day` (summed across all stops)
# is below this threshold — keeps the dashboard focused on routes with
# enough observations to be statistically meaningful. Set to None to disable.
MIN_ARRIVALS_PER_ROUTE = 5000
MAX_TRIPS_PER_PERIOD = 12

# ── NEW: path simplification ──────────────────────────────────────────
# Douglas–Peucker tolerance, in degrees (CRS is EPSG:4326). Roughly:
#   1e-4 ≈ 11 m, 5e-5 ≈ 5 m, 1e-5 ≈ 1 m.
PATH_SIMPLIFY_TOL = 1e-5
# Coordinate rounding for the JSON payload.
PATH_COORD_DECIMALS = 5

# ── Period mapping (short ↔ long names from the v2 aggregator) ──────────
PERIOD_COLS = {
    'all':     'day',
    'am_off':  'morning_offpeak',
    'am_peak': 'morning_peak',
    'midday':  'midday_offpeak',
    'pm_peak': 'evening_peak',
    'pm_off':  'evening_offpeak',
}
DETAIL_PERIODS = [k for k in PERIOD_COLS if k != 'all']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"input  : {INPUT_DIR}")
print(f"output : {OUTPUT_DIR}")
print(f"filters: route_type_name={ALLOWED_ROUTE_TYPES}, "
      f"N_ROUTES={'all' if N_ROUTES is None else N_ROUTES}, "
      f"min_arrivals={MIN_ARRIVALS_PER_ROUTE}")
print(f"path   : simplify_tol={PATH_SIMPLIFY_TOL} deg, decimals={PATH_COORD_DECIMALS}")

input  : D:\2026_03-Bus_Project\output\aggregate
output : C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay
filters: route_type_name=['Bus'], N_ROUTES=all, min_arrivals=5000
path   : simplify_tol=1e-05 deg, decimals=5


## 1. Load and filter stop-metrics CSVs

Identical to v2.

In [2]:
def load_metrics(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path,
                     dtype={"route_id": str, "direction_id": str, "stop_id": str},
                     low_memory=False)
    if ALLOWED_ROUTE_TYPES is not None:
        before = len(df)
        df = df[df["route_type_name"].isin(ALLOWED_ROUTE_TYPES)]
        print(f"  {path.name}: {before:,} → {len(df):,} rows after type filter")
    else:
        print(f"  {path.name}: {len(df):,} rows (no type filter)")
    df["stop_name"] = df["stop_name"].fillna("")
    return df.reset_index(drop=True)


df_wd = load_metrics(WEEKDAY_CSV)
df_we = load_metrics(WEEKEND_CSV)
if df_wd.empty:
    raise RuntimeError(
        f"No weekday rows survived the route_type_name filter "
        f"{ALLOWED_ROUTE_TYPES}. Check ALLOWED_ROUTE_TYPES.")

print(f"\nweekday: {len(df_wd):,} rows · weekend: {len(df_we):,} rows")
print(f"unique route_short_name (weekday): {df_wd['route_short_name'].nunique()}")

  stop_metrics_weekday_v2.csv: 44,935 → 44,935 rows after type filter
  stop_metrics_weekend_v2.csv: 22,513 → 22,513 rows after type filter

weekday: 44,935 rows · weekend: 22,513 rows
unique route_short_name (weekday): 574


## 2. Load route geometries and ordered stop sequences

Reads the `routes` layer from `stop_metrics_v2.gpkg` (per-`(route_id,
direction_id)` LineStrings + a JSON-encoded `stop_sequence` ordered by
`stop_frac`). Unlike v2/v3 we **keep every direction**: each row will
become its own record in the output.

In [3]:
gdf_routes = gpd.read_file(STOP_METRICS_GPKG, layer="routes")

if ALLOWED_ROUTE_TYPES is not None:
    before = len(gdf_routes)
    gdf_routes = gdf_routes[gdf_routes["route_type_name"].isin(ALLOWED_ROUTE_TYPES)]
    print(f"  routes layer: {before:,} -> {len(gdf_routes):,} rows after type filter")
else:
    print(f"  routes layer: {len(gdf_routes):,} rows")

if gdf_routes.crs is not None and gdf_routes.crs.to_epsg() != 4326:
    gdf_routes = gdf_routes.to_crs(4326)

gdf_routes["stop_sequence"] = gdf_routes["stop_sequence"].apply(
    lambda s: json.loads(s) if isinstance(s, str)
    else (list(s) if s is not None else [])
)

# Index by (route_short_name, direction_id). If duplicates exist (multiple
# route_id values share the same short_name + direction), keep the one with
# the most stops in the study area.
gdf_routes = (gdf_routes.sort_values("n_stops_in_area", ascending=False)
                        .drop_duplicates(["route_short_name", "direction_id"],
                                         keep="first"))
routes_idx = gdf_routes.set_index(["route_short_name", "direction_id"])

print(f"\n{len(routes_idx):,} (route_short_name, direction_id) pairs available")
print(routes_idx[["route_id", "n_stops_in_area", "geometry_method"]].head().to_string())

  routes layer: 1,305 → 1,179 rows after type filter

591 short_names have a canonical stop_sequence + geometry
                 route_id direction_id  n_stops_in_area geometry_method
route_short_name                                                       
41                  92514            0              118           mixed
356                  9947            1              114           mixed
35                  81812            1              113           mixed
343                 94552            1              108           mixed
149                 92592            0              106           mixed


## 3. Rank `(route, direction)` pairs and pick the top N

Each direction is ranked separately by total weekday `n_arrivals_day`.
`MIN_ARRIVALS_PER_ROUTE` and `N_ROUTES` apply to (route, direction)
pairs, not to short_names.

In [4]:
arrivals_by_dir = (df_wd.groupby(['route_short_name', 'direction_id'])
                       ['n_arrivals_day']
                       .sum()
                       .sort_values(ascending=False))

n_total = len(arrivals_by_dir)
if MIN_ARRIVALS_PER_ROUTE is not None:
    arrivals_by_dir = arrivals_by_dir[arrivals_by_dir >= MIN_ARRIVALS_PER_ROUTE]
    print(f"min_arrivals filter: {n_total:,} -> {len(arrivals_by_dir):,} "
          f"(route, direction) pairs (threshold {MIN_ARRIVALS_PER_ROUTE:,})")

if N_ROUTES is None:
    selected_pairs = list(arrivals_by_dir.index)
    print(f"all {len(selected_pairs):,} (route, direction) pairs selected")
else:
    selected_pairs = list(arrivals_by_dir.index[:N_ROUTES])
    print(f"top {len(selected_pairs)} of {len(arrivals_by_dir):,} pairs selected")

print(f"first 5: {selected_pairs[:5]}")

min_arrivals filter: 574 → 183 routes (threshold 5,000 weekday arrivals)
all 183 routes selected (no N_ROUTES limit)
first 10: ['192', '36', '163', '17', '84', '135', '471', '201', '409', '52']


## 4. Build `network_routes.json`

For each selected `(route_short_name, direction_id)` pair we emit the
ordered stop list, the simplified routed `path`, terminus labels, and
per-period weighted stats. Periods with zero observed arrivals return
`mean=None, otp=None, n=0` so the dashboard can distinguish
"no service" from "0% on-time".

In [5]:
# Clamp for nonsensical per-stop medians (missed-trip 1440-min entries
# from the source feed). Anything beyond this is treated as missing data.
OUTLIER_MIN = 60.0


def weighted_median(values, weights):
    """Weighted median. NaN values are dropped (and their weight ignored)."""
    s = pd.DataFrame({"v": values, "w": weights}).dropna(subset=["v"])
    s = s[s["w"] > 0].sort_values("v")
    total = s["w"].sum()
    if total == 0:
        return None
    cw = s["w"].cumsum()
    return float(s.loc[cw >= total / 2, "v"].iloc[0])


def weighted_period_stats(stops_df: pd.DataFrame, period_col: str) -> dict:
    """Route-level stats. Outlier-robust: median of per-stop Q2 weighted by
    arrivals, plus arrivals-true on-time ratio. None when no data."""
    n_col   = f"n_arrivals_{period_col}"
    q2_col  = f"Q2_{period_col}"
    otp_col = f"pct_on_time_{period_col}"

    n = stops_df[n_col].fillna(0).astype(float)
    q2 = stops_df[q2_col].where(stops_df[q2_col].abs() <= OUTLIER_MIN)
    otp = stops_df[otp_col]
    total_n = int(n.sum())
    if total_n == 0:
        return {"median": None, "otp": None, "n": 0}

    median = weighted_median(q2, n)

    w_otp = n.where(otp.notna(), 0)
    otp_val = (float((otp.fillna(0) * w_otp).sum() / w_otp.sum())
               if w_otp.sum() else None)
    return {"median": median, "otp": otp_val, "n": total_n}


def busiest_by_stop_per_dir(df: pd.DataFrame) -> pd.DataFrame:
    """Dedupe on (route_short_name, direction_id, stop_id), keeping busiest."""
    return (df.sort_values("n_arrivals_day", ascending=False)
              .drop_duplicates(["route_short_name", "direction_id", "stop_id"]))


def geom_to_path(geom, tol=PATH_SIMPLIFY_TOL, decimals=PATH_COORD_DECIMALS):
    if geom is None or geom.is_empty:
        return []
    if tol and tol > 0:
        geom = geom.simplify(tol, preserve_topology=False)
    if isinstance(geom, LineString):
        coords = list(geom.coords)
    elif isinstance(geom, MultiLineString):
        coords = []
        for part in geom.geoms:
            coords.extend(list(part.coords))
    else:
        return []
    return [[round(float(y), decimals), round(float(x), decimals)] for x, y in coords]


df_wd_busy = busiest_by_stop_per_dir(df_wd)

network = []
skipped_no_seq  = 0
skipped_too_few = 0
path_vertex_total = 0

for short_name, direction_id in selected_pairs:
    if (short_name, direction_id) not in routes_idx.index:
        skipped_no_seq += 1
        continue
    row = routes_idx.loc[(short_name, direction_id)]
    seq = row["stop_sequence"]
    if not isinstance(seq, list) or len(seq) < MIN_STOPS_PER_ROUTE:
        skipped_too_few += 1
        continue

    sub = (df_wd_busy[(df_wd_busy["route_short_name"] == short_name) &
                      (df_wd_busy["direction_id"]     == direction_id)]
                     .set_index("stop_id"))
    stops_out = []
    for sid in seq:
        if sid not in sub.index:
            continue
        r = sub.loc[sid]
        stops_out.append({
            "id":   sid,
            "name": r["stop_name"],
            "lat":  round(float(r["lat"]), 5),
            "lon":  round(float(r["lon"]), 5),
        })
    if len(stops_out) < MIN_STOPS_PER_ROUTE:
        skipped_too_few += 1
        continue

    path = geom_to_path(row["geometry"])
    path_vertex_total += len(path)

    rows_out = sub.loc[[s["id"] for s in stops_out]]
    terminus_from = stops_out[0]["name"]
    terminus_to   = stops_out[-1]["name"]
    network.append({
        "key":           f"{short_name}-{direction_id}",
        "name":          str(short_name),
        "route_id":      str(row["route_id"]) if pd.notna(row["route_id"]) else None,
        "direction_id":  str(direction_id) if pd.notna(direction_id) else None,
        "direction":     f"{terminus_from} -> {terminus_to}",
        "terminus_from": terminus_from,
        "terminus_to":   terminus_to,
        "stops":         stops_out,
        "path":          path,
        "periods":       {k: weighted_period_stats(rows_out, col)
                          for k, col in PERIOD_COLS.items()},
    })

print(f"network: {len(network)} (route, direction) records "
      f"(skipped: {skipped_no_seq} no-sequence, {skipped_too_few} too-few-stops)")
if network:
    avg_stops = sum(len(r["stops"]) for r in network) / len(network)
    avg_path  = path_vertex_total / len(network)
    print(f"mean stops per record: {avg_stops:.1f}")
    print(f"mean path vertices:    {avg_path:.1f}")
    print(f"unique short_names:    {len({r['name'] for r in network})}")

network: 183 routes (skipped: 0 no-sequence, 0 too-few-stops)
mean stops per route: 56.6
mean path vertices:   213.9
total path vertices:  39,145


## 5. Build `route_details.json`

Keyed by the same `key` (`<short_name>-<direction_id>`) used in
`network_routes.json` so consumers can look up stop-level detail
without ambiguity. Per-period per-stop records are unchanged from v3.

In [6]:
def parse_delay_list(raw):
    if pd.isna(raw):
        return []
    try:
        return [round(float(x), 2) for x in ast.literal_eval(raw)]
    except (ValueError, SyntaxError):
        return []


def even_sample(arr, k):
    if len(arr) <= k:
        return arr
    idx = np.linspace(0, len(arr) - 1, k).round().astype(int)
    return [arr[i] for i in idx]


def stop_period_record(row, period_col):
    n = row.get(f"n_arrivals_{period_col}", 0)
    if pd.isna(n) or n == 0:
        return {"median": None, "otp": None, "n": 0, "d": []}
    delays = even_sample(parse_delay_list(row.get(f"list_delay_{period_col}")),
                         MAX_TRIPS_PER_PERIOD)
    q2  = row[f"Q2_{period_col}"]
    otp = row[f"pct_on_time_{period_col}"]
    # Treat extreme per-stop medians (missed-trip codes) as no signal.
    q2_clean = None if pd.isna(q2) or abs(float(q2)) > OUTLIER_MIN else round(float(q2), 2)
    return {
        "median": q2_clean,
        "otp":    None if pd.isna(otp) else round(float(otp), 1),
        "n":      int(n),
        "d":      delays,
    }


def empty_period_block():
    return {p: {"median": None, "otp": None, "n": 0, "d": []} for p in DETAIL_PERIODS}


df_we_busy = busiest_by_stop_per_dir(df_we)

details = {}
for r in network:
    sn  = r["name"]
    did = r["direction_id"]
    rows_wd = (df_wd_busy[(df_wd_busy["route_short_name"] == sn) &
                          (df_wd_busy["direction_id"]     == did)]
                         .set_index("stop_id"))
    rows_we = (df_we_busy[(df_we_busy["route_short_name"] == sn) &
                          (df_we_busy["direction_id"]     == did)]
                         .set_index("stop_id"))
    by_stop = {}
    for stop in r["stops"]:
        sid = stop["id"]
        wd_row = rows_wd.loc[sid] if sid in rows_wd.index else None
        we_row = rows_we.loc[sid] if sid in rows_we.index else None
        by_stop[sid] = {
            "wd": ({p: stop_period_record(wd_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                   if wd_row is not None else empty_period_block()),
            "we": ({p: stop_period_record(we_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                   if we_row is not None else empty_period_block()),
        }
    details[r["key"]] = by_stop

print(f"detail records for {len(details)} (route, direction) pairs")

detail records for 183 routes


## 6. Write the JSON files

In [7]:
summary_path = OUTPUT_DIR / 'network_routes.json'
details_path = OUTPUT_DIR / 'route_details.json'

with summary_path.open('w', encoding='utf-8') as f:
    json.dump({'routes': network}, f, separators=(',', ':'))
with details_path.open('w', encoding='utf-8') as f:
    json.dump(details, f, separators=(',', ':'))

for p in (summary_path, details_path):
    print(f'{p.name}: {p.stat().st_size / 1024:.1f} KB  ->  {p}')

network_routes.json: 1629.3 KB  ->  C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay\network_routes.json
route_details.json: 9446.6 KB  ->  C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay\route_details.json


In [8]:
# Quick sanity check on one record.
if network:
    r = network[0]
    print(f"key={r['key']}  name={r['name']}  dir={r['direction_id']}")
    print(f"  {r['direction']}")
    print(f"  {len(r['stops'])} stops · {len(r['path'])} path vertices")
    for k, v in r['periods'].items():
        m = '   --- ' if v['median'] is None else f"{v['median']:7.2f}"
        o = '  -- '   if v['otp']    is None else f"{v['otp']:5.1f}"
        print(f"  {k:8s} median={m} min  otp={o}%  n={v['n']}")
else:
    print("(no records - check filters)")

route 192 · dir=1 (Hazel Grove Park & Ride → Piccadilly Gardens) · 57 stops · 181 path vertices
  all      mean=   0.82 min  otp= 28.3%  n=78903
  am_off   mean=  12.72 min  otp= 32.3%  n=8694
  am_peak  mean=  -0.13 min  otp= 32.8%  n=8713
  midday   mean=   0.33 min  otp= 28.5%  n=30253
  pm_peak  mean=   1.89 min  otp= 20.0%  n=15253
  pm_off   mean=   1.22 min  otp= 30.9%  n=15990

first 3 path verts (lat, lon): [[53.37369, -2.11325], [53.37511, -2.11366], [53.37558, -2.11388]]
last 3:                        [[53.47942, -2.23507], [53.47977, -2.23553], [53.48066, -2.23511]]
